#### Import Pandas Library

In [1]:
import pandas as pd
import openpyxl
import random
from datetime import datetime,timedelta

#### Create a excel workbook with 3 sheets (Customer,Order,Region)

##### 1) Create data generation functions of dictonary type

In [2]:
names = ["Sujay Kumar","Raj Singh","Keshav Sinha","Ashutosh Dash","Piyush Kumar"]
cities=["Bhubaneswar","Bhagalpur","Pune","Mumbai","Hyderabad"]

def generate_customer_data(row:int)->dict:
    return {
        "customer_id":id(row),
        "customer_name":random.choice(names),
        "payment_status":random.choice(["Paid","Not Paid"])
        }


def generate_order_data(row:int)->dict:
    return {
        "customer_id":id(row),
        "order_date":(datetime(2026,10,1)-timedelta(days=random.randint(0,2000))).strftime("%Y-%m-%d")
    }

def generate_region_data(row:int)->dict:
    return {
        "customer_id":id(row),
        "city":random.choice(cities)
    }

##### 2) Create a function that convert list of dictonary object to DataFrame 

In [3]:
from pandas import DataFrame
def generate_dataFrame(data:list)-> DataFrame:
    return pd.DataFrame(data)


##### 3) Generate the customer details workbook

In [4]:
def create_excel_cust_workbook():
    FILE_NAME = "Customer_details.xlsx"
    with pd.ExcelWriter(FILE_NAME,engine = "openpyxl") as writer:
        customer_data = [generate_customer_data(i) for i in range(1,51)]
        order_data = [generate_order_data(i) for i in range(1,51)]
        region_data = [generate_order_data(i) for i in range(1,51)]
        cust_df = generate_dataFrame(customer_data)
        order_df = generate_dataFrame(order_data)
        region_df = generate_dataFrame(region_data)
        cust_df.to_excel(writer,sheet_name="Customers",index=False)
        order_df.to_excel(writer,sheet_name="Orders",index=False)
        region_df.to_excel(writer,sheet_name="Region",index=False)
    print(f"{FILE_NAME} excel file created successfully")


create_excel_cust_workbook()

Customer_details.xlsx excel file created successfully


#### Pyspark Practice

##### Install and import Pyspark Library

In [5]:
pip install pyspark --break-system-packages

Note: you may need to restart the kernel to use updated packages.


In [6]:
pip show pyspark

Name: pyspark
Version: 4.2.0
Summary: Apache Spark Python API
Home-page: https://github.com/apache/spark/tree/master/python
Author: Spark Developers
Author-email: dev@spark.apache.org
License: Apache-2.0
Location: C:\Users\sujaykumar\AppData\Local\anaconda3\Lib\site-packages
Requires: py4j
Required-by: 
Note: you may need to restart the kernel to use updated packages.


##### Create pyspark instance

In [7]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
print(spark)

#### Q1 Read an Excel Workbook with Multiple sheets as Input and Output (with Headers)

In [8]:
import pandas as pd
customer_sheet=pd.read_excel("Customer_details.xlsx",sheet_name="Customers")
customer_sheet.head(5)

,customer_id,customer_name,payment_status
0,140704575300520,Keshav Sinha,Not Paid
1,140704575300552,Keshav Sinha,Paid
2,140704575300584,Keshav Sinha,Not Paid
3,140704575300616,Sujay Kumar,Not Paid
4,140704575300648,Raj Singh,Paid


In [9]:
order_sheet=pd.read_excel("Customer_details.xlsx",sheet_name="Orders")
order_sheet.head(5)

,customer_id,order_date
0,140704575300520,2026-04-29
1,140704575300552,2025-03-05
2,140704575300584,2022-09-02
3,140704575300616,2022-03-22
4,140704575300648,2023-06-20


In [10]:
region_sheet=pd.read_excel("Customer_details.xlsx",sheet_name="Region")
region_sheet.head(5)

,customer_id,order_date
0,140704575300520,2026-09-17
1,140704575300552,2022-05-11
2,140704575300584,2026-03-02
3,140704575300616,2021-10-10
4,140704575300648,2024-01-05


In [11]:
cust_df=spark.createDataFrame(customer_sheet)
cust_df.show(5)

order_df=spark.createDataFrame(order_sheet)
order_df.show(5)

region_df=spark.createDataFrame(region_sheet)
region_df.show(5)

+---------------+-------------+--------------+
|    customer_id|customer_name|payment_status|
+---------------+-------------+--------------+
|140704575300520| Keshav Sinha|      Not Paid|
|140704575300552| Keshav Sinha|          Paid|
|140704575300584| Keshav Sinha|      Not Paid|
|140704575300616|  Sujay Kumar|      Not Paid|
|140704575300648|    Raj Singh|          Paid|
+---------------+-------------+--------------+
only showing top 5 rows
+---------------+----------+
|    customer_id|order_date|
+---------------+----------+
|140704575300520|2026-04-29|
|140704575300552|2025-03-05|
|140704575300584|2022-09-02|
|140704575300616|2022-03-22|
|140704575300648|2023-06-20|
+---------------+----------+
only showing top 5 rows
+---------------+----------+
|    customer_id|order_date|
+---------------+----------+
|140704575300520|2026-09-17|
|140704575300552|2022-05-11|
|140704575300584|2026-03-02|
|140704575300616|2021-10-10|
|140704575300648|2024-01-05|
+---------------+----------+
only sh

#### Q2 Insert a new column: Use another method instead of loc


In [12]:
from pyspark.sql.functions import lit
new_cust_df=cust_df.withColumn('Country',lit('India'))
new_cust_df.show(5)

+---------------+-------------+--------------+-------+
|    customer_id|customer_name|payment_status|Country|
+---------------+-------------+--------------+-------+
|140704575300520| Keshav Sinha|      Not Paid|  India|
|140704575300552| Keshav Sinha|          Paid|  India|
|140704575300584| Keshav Sinha|      Not Paid|  India|
|140704575300616|  Sujay Kumar|      Not Paid|  India|
|140704575300648|    Raj Singh|          Paid|  India|
+---------------+-------------+--------------+-------+
only showing top 5 rows


#### Q3 Drop a new column: Drop multiple columns given as input list

In [13]:
to_be_dropped = ["customer_name","Country","payment_status"]
drop_cust_df = new_cust_df.drop(*to_be_dropped)
print("After dropping 3 columns")
drop_cust_df.show(5)

After dropping 3 columns
+---------------+
|    customer_id|
+---------------+
|140704575300520|
|140704575300552|
|140704575300584|
|140704575300616|
|140704575300648|
+---------------+
only showing top 5 rows


#### Q4 Fill already existing column with a single value: Update 2 different columns using conditional statements


In [14]:
from pyspark.sql.functions import when,col
update_cust_df=cust_df.withColumn('Copoun_code',when(col('payment_status') == 'Paid', lit(2192)).otherwise(lit(0)))
update_cust_df.show(5)

+---------------+-------------+--------------+-----------+
|    customer_id|customer_name|payment_status|Copoun_code|
+---------------+-------------+--------------+-----------+
|140704575300520| Keshav Sinha|      Not Paid|          0|
|140704575300552| Keshav Sinha|          Paid|       2192|
|140704575300584| Keshav Sinha|      Not Paid|          0|
|140704575300616|  Sujay Kumar|      Not Paid|          0|
|140704575300648|    Raj Singh|          Paid|       2192|
+---------------+-------------+--------------+-----------+
only showing top 5 rows


#### Q5 Change the name of a single column to something new

In [15]:
renamed_col_cust_df=cust_df.withColumnRenamed('customer_name','Name')
renamed_col_cust_df.printSchema()
renamed_col_cust_df.show(2)

root
 |-- customer_id: long (nullable = true)
 |-- Name: string (nullable = true)
 |-- payment_status: string (nullable = true)

+---------------+------------+--------------+
|    customer_id|        Name|payment_status|
+---------------+------------+--------------+
|140704575300520|Keshav Sinha|      Not Paid|
|140704575300552|Keshav Sinha|          Paid|
+---------------+------------+--------------+
only showing top 2 rows


#### Q6 Add prefix to a list of columns


In [16]:
prefix ='Boehringer_'
colums=['customer_id','customer_name','payment_status','customer_name']
prefixed_df=update_cust_df
for cols in colums:
    prefixed_df = prefixed_df.withColumnRenamed(cols,prefix+cols)

prefixed_df.show(5)
prefixed_df.printSchema()

+----------------------+------------------------+-------------------------+-----------+
|Boehringer_customer_id|Boehringer_customer_name|Boehringer_payment_status|Copoun_code|
+----------------------+------------------------+-------------------------+-----------+
|       140704575300520|            Keshav Sinha|                 Not Paid|          0|
|       140704575300552|            Keshav Sinha|                     Paid|       2192|
|       140704575300584|            Keshav Sinha|                 Not Paid|          0|
|       140704575300616|             Sujay Kumar|                 Not Paid|          0|
|       140704575300648|               Raj Singh|                     Paid|       2192|
+----------------------+------------------------+-------------------------+-----------+
only showing top 5 rows
root
 |-- Boehringer_customer_id: long (nullable = true)
 |-- Boehringer_customer_name: string (nullable = true)
 |-- Boehringer_payment_status: string (nullable = true)
 |-- Copoun_co

#### Q7 Filter records with any value: Use Multiple filter conditions


##### Filter paid and unpaid customer records

In [35]:
from pyspark.sql.functions import *
paid_df=cust_df.filter(col('payment_status') == 'Paid')


paid_df=paid_df.withColumnRenamed('payment_status','payment_status_p')
paid_df.show(5)

unpaid_df = cust_df.filter(col('payment_status') == 'Not Paid')
unpaid_df = unpaid_df.withColumnRenamed('payment_status','payment_status_np')
unpaid_df.show(5)

+---------------+-------------+----------------+
|    customer_id|customer_name|payment_status_p|
+---------------+-------------+----------------+
|140704575300552| Keshav Sinha|            Paid|
|140704575300648|    Raj Singh|            Paid|
|140704575300744|  Sujay Kumar|            Paid|
|140704575300808| Piyush Kumar|            Paid|
|140704575300840| Piyush Kumar|            Paid|
+---------------+-------------+----------------+
only showing top 5 rows
+---------------+-------------+-----------------+
|    customer_id|customer_name|payment_status_np|
+---------------+-------------+-----------------+
|140704575300520| Keshav Sinha|         Not Paid|
|140704575300584| Keshav Sinha|         Not Paid|
|140704575300616|  Sujay Kumar|         Not Paid|
|140704575300680|    Raj Singh|         Not Paid|
|140704575300712|    Raj Singh|         Not Paid|
+---------------+-------------+-----------------+
only showing top 5 rows


#### Q8 Joining 2 Tables

##### Joining customer table with order table

In [20]:
cust_df.show(2)
order_df.show(2)

joined_df = cust_df.join(order_df,on=cust_df['customer_id']==order_df['customer_id'],how='inner')
joined_df.show(10)

+---------------+-------------+--------------+
|    customer_id|customer_name|payment_status|
+---------------+-------------+--------------+
|140704575300520| Keshav Sinha|      Not Paid|
|140704575300552| Keshav Sinha|          Paid|
+---------------+-------------+--------------+
only showing top 2 rows
+---------------+----------+
|    customer_id|order_date|
+---------------+----------+
|140704575300520|2026-04-29|
|140704575300552|2025-03-05|
+---------------+----------+
only showing top 2 rows
+---------------+-------------+--------------+---------------+----------+
|    customer_id|customer_name|payment_status|    customer_id|order_date|
+---------------+-------------+--------------+---------------+----------+
|140704575300520| Keshav Sinha|      Not Paid|140704575300520|2026-04-29|
|140704575300552| Keshav Sinha|          Paid|140704575300552|2025-03-05|
|140704575300584| Keshav Sinha|      Not Paid|140704575300584|2022-09-02|
|140704575300616|  Sujay Kumar|      Not Paid|140704

#### Q9 Union 2 Table

##### Union Paid dataframe with unpaid customer dataframe

In [40]:
union_df = paid_df.unionByName(unpaid_df,allowMissingColumns=True)
union_df.show(50)

+---------------+-------------+----------------+-----------------+
|    customer_id|customer_name|payment_status_p|payment_status_np|
+---------------+-------------+----------------+-----------------+
|140704575300552| Keshav Sinha|            Paid|             NULL|
|140704575300648|    Raj Singh|            Paid|             NULL|
|140704575300744|  Sujay Kumar|            Paid|             NULL|
|140704575300808| Piyush Kumar|            Paid|             NULL|
|140704575300840| Piyush Kumar|            Paid|             NULL|
|140704575300872|Ashutosh Dash|            Paid|             NULL|
|140704575301000|    Raj Singh|            Paid|             NULL|
|140704575301064|Ashutosh Dash|            Paid|             NULL|
|140704575301128|    Raj Singh|            Paid|             NULL|
|140704575301160| Keshav Sinha|            Paid|             NULL|
|140704575301192|  Sujay Kumar|            Paid|             NULL|
|140704575301224|Ashutosh Dash|            Paid|             N